# Physical projection: 2601_XPCS
Run 00_prepare first. Shared cell structure for both paper cases.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import h5py
import matplotlib.pyplot as plt
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/source_manifest.json").is_file())
sys.path.insert(0, str(ROOT))
from library import phase_retrieval_universal as universal
from library import real_space_constraints as real_space
from library import magnitude_constraints as magnitude
from library.phase_retrieval_backend import to_numpy
CACHE = ROOT / "data/2601_XPCS/prepared.h5"
OUTPUT = ROOT / "results/hysteresis.h5"
SATURATED_POINTS = {}  # e.g. {(263, 10): +1., (263, 34): -1.}; verify independently.
POINT_INDICES = [10,12,14,15,17,20,23,27,28,29,30,31,32,34]
with h5py.File(CACHE) as handle:
    config=json.loads(handle.attrs['config_json'])
    labels=handle['state'].asstr()[:].tolist()
    keys=list(zip(handle['scan'][()].tolist(),handle['point'][()].tolist()))
    indices=list(POINT_INDICES) if POINT_INDICES is not None else list(range(len(labels)))
    for key in SATURATED_POINTS:
        if key not in keys: raise ValueError(f"Missing anchor {key}")
        if keys.index(key) not in indices: indices.append(keys.index(key))
    if len(set(indices)) != len(indices) or any(i<0 or i>=len(labels) for i in indices):
        raise ValueError("Invalid acquisition indices")
    indices=sorted(indices)  # Preserve acquisition order, including repeated currents.
    holograms=np.stack([handle['holograms'][i] for i in indices])
    mask_pixel=np.stack([handle['mask_pixel'][i] for i in indices])
    support=handle['supportmask'][()]; material=handle['material_mask'][()].astype(float)
    current=handle['current_A'][()][indices]
    energies=handle['energy_ev'][()][indices]
    state=[labels[i] for i in indices]
if np.ptp(energies)>.25: raise ValueError("Expected one photon energy")
energy=[float(np.median(energies))]*len(indices)
polarizations=[1.]*len(indices)
saturated={labels[keys.index(key)]:value for key,value in SATURATED_POINTS.items()}
crop, binning=config['crop'],config['binning']


## 1. Validate inputs and metadata

In [ ]:
assert holograms.ndim==3 and mask_pixel.shape==holograms.shape
assert support.shape==holograms.shape[-2:] and material.shape==support.shape
assert len(state)==len(energy)==len(polarizations)==len(holograms)
assert np.all(np.isfinite(holograms)) and np.any(support)
assert set(saturated.values()) <= {-1.,1.}
illumination=["beam0"]*len(holograms)
metadata=dict(state=state, energy=energy,
              polarization=polarizations, illumination=illumination)
print("Observations",len(holograms),"energies",sorted(set(energy)),"anchors",saturated)
fig,ax=plt.subplots(1,2)
ax[0].imshow(np.log1p(np.maximum(holograms[0],0))); ax[0].set_title('First detector image')
ax[1].imshow(support); ax[1].set_title('Source support')


## 2. Set the physical model and iteration schedule

In [ ]:
# Shared controls. Change these explicitly for a new dataset.
MODES=[1,2]
RL=False  # Append a second 700 HAPRE + 50 ER round with RL fitting.
PHYSICAL_EVERY=50
PHYSICAL_IT=50
recipe=universal.default_universal_phase_retrieval_recipe(RL=RL)
recipe.update(**metadata, modes=MODES,
    crop=crop, binning=binning, roi=None,
    recenter_modal_supports=True, mode_support_center="image",
    material_thickness=material, fit_material_thickness=False,
    physical_projection_object_roi=True, physical_phase_reference=True,
    saturated_states=saturated or None,
    mode_initialization="support_fft", 
    projection_relaxation=1.,
    constrain_nonphysical_modes_common=True,
    projection_constraints_inside_support_only=True, zero_magnetization_outside_support=True,
    clip_magnetization=False, magnetization_trend="none", bound_magnetization_median=False,
    physical_spatial_weighting="exit_wave_intensity", random_seed=0,
    fft_workers=1)
for stage in recipe['stages']:
    stage.update(physical_it=PHYSICAL_IT, physical_every=PHYSICAL_EVERY)
# Final model fields satisfy the physical projection; detector agreement is measured below.
recipe.update(energy_values=np.asarray(sorted(set(energy))),
    align_global_phase=True, fit_illumination_intensity=True,
    fit_aperture_illumination=True, object_illumination_mode="linear_plane")


## 3. Reconstruction metadata

In [ ]:
# All observation metadata and numerical choices are contained in recipe.


## 4. Joint reconstruction

In [ ]:
reconstruction=universal.initialize_phase_retrieval(holograms, mask_pixel, support, recipe)
# Geometry and masks are prepared once; the main loop applies only constraints.
measured=reconstruction['amplitudes']**2
support_used=reconstruction['support']
fields, initial, components, bsmasks, errors=universal.universal_phase_retrieval_algorithm(reconstruction)
recipe=reconstruction['recipe']
print("Physical model residual",components.get('fit_residual_rms'))


## 5. Compare detector residuals and inspect maps

In [ ]:
# Match the same detector model used in the last retrieval iteration.
modal_fields=fields[:,None] if fields.ndim==3 else fields
kernels=components.get('RL_kernels') if reconstruction['schedule'][-1]['RL'] else None
prediction=to_numpy(magnitude.modeled_intensity(modal_fields,kernels))
valid=~np.asarray(bsmasks,bool)
target=np.where(valid,measured,0)
residual=np.where(valid,prediction-measured,0)
relative=np.linalg.norm(residual,axis=(-2,-1))/np.maximum(np.linalg.norm(target,axis=(-2,-1)),1e-30)
detector_errors={'final':relative}
fig,ax=plt.subplots()
ax.plot(relative,'o-')
ax.set(xlabel='Observation in acquisition order',ylabel='Relative intensity residual')
magnetization=np.asarray(components['magnetization'])
fig,ax=plt.subplots(); im=ax.imshow(magnetization[0],cmap='RdBu_r'); fig.colorbar(im,ax=ax)
ax.set_title('First fitted state: magnetization factor')


## 6. Extract the paper observables

In [ ]:
# Analyze fitted maps and the final exit wave in the physical-projection frame.
material_used=reconstruction['material_thickness']>0
ROI=material_used  # Replace with a documented boolean ROI for local loops.
if not np.any(ROI): raise ValueError('Empty analysis ROI')
state_names=list(components['state_names'])
state_currents=np.asarray([current[state.index(name)] for name in state_names])
fit_curve=np.mean(magnetization[:,ROI],axis=1)
paper_analysis={'current_A':state_currents,'model_curve':fit_curve,'roi_mask':ROI}
fig,ax=plt.subplots()
ax.plot(state_currents,fit_curve,'o-',label='Physical model')
positive=next((i for i,s in enumerate(state) if saturated.get(s)==1),None)
negative=next((i for i,s in enumerate(state) if saturated.get(s)==-1),None)
if positive is not None and negative is not None:
    primary=fields[:,0] if fields.ndim==4 else fields
    obj=np.fft.fft2(np.fft.fftshift(primary,axes=(-2,-1)),axes=(-2,-1))
    direction=np.mean((obj[positive]-obj[negative])[ROI])
    if np.isfinite(direction) and abs(direction)>1e-12:
        mean=np.mean(np.real(obj*np.conj(direction)/abs(direction))[:,ROI],axis=1)
        span=mean[positive]-mean[negative]
        if np.isfinite(span) and abs(span)>1e-12:
            curve=2*(mean-(mean[positive]+mean[negative])/2)/span
            paper_analysis['exit_wave_current_A']=np.asarray(current)
            paper_analysis['exit_wave_endpoint_curve']=curve
            ax.plot(current,curve,'o--',label='Final exit-wave endpoint contrast')
else:
    print('No two-sign saturation calibration: model curve is a relative factor.')
ax.set(xlabel='Magnet current (A)',ylabel='Magnetization / relative contrast'); ax.legend()
# Endpoint contrast is derived from the same fit, not independent calibration.


## 7. Save data, settings, and the figure

In [ ]:
# Store arrays and nested numerical metadata directly; this small writer stays visible here.
def save_tree(group, values):
    for key,value in values.items():
        if isinstance(value,dict): save_tree(group.create_group(str(key)),value)
        elif value is None: group.attrs[str(key)]='None'
        else:
            array=np.asarray(value)
            if array.dtype.kind in 'biufc': group.create_dataset(str(key),data=array)
            else: group.attrs[str(key)]=json.dumps(value,default=lambda x: np.asarray(x).tolist())
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
if OUTPUT.exists(): raise FileExistsError(f'Choose a new OUTPUT path: {OUTPUT}')
with h5py.File(OUTPUT,'x') as handle:
    for name,value in dict(fields=fields,initial_fields=initial,holograms=measured,
                           mask_pixel=bsmasks,supportmask=support_used).items():
        handle.create_dataset(name,data=value,compression='gzip')
    save_tree(handle.create_group('metadata'),metadata)
    save_tree(handle.create_group('preprocessing'),config)
    save_tree(handle.create_group('recipe'),recipe)
    save_tree(handle.create_group('components'),components)
    save_tree(handle.create_group('errors'),errors)
    save_tree(handle.create_group('detector_errors'),detector_errors)
    save_tree(handle.create_group('paper_analysis'),paper_analysis)
fig.savefig(OUTPUT.with_suffix('.pdf'),bbox_inches='tight')
print(OUTPUT)
